# MEER Script-to-Animation — free-first Colab prototype

Upload a character reference image, paste a script, generate one LTX image-to-video clip per script line, then stitch clips and create narration/subtitles.

Free GPU access is limited; no paid API is called. Start with 2–3 lines to test. Character reference improves consistency but cannot guarantee identical pixels or perfect scene interpretation. No GitHub token is needed; outputs stay in the Colab session.

How to use: open in Google Colab, select GPU if available, edit SCRIPT and CHARACTER settings, run all, upload one clean character reference when prompted, then download the final MP4, narration MP3 and subtitle SRT.

In [ ]:
# ---- EDIT THESE SETTINGS ----
HERO = "Rayo"
CHARACTER_DESCRIPTION = "a sleek red 3D cartoon race car with a yellow lightning bolt mark, windshield eyes and a friendly bumper smile"
STYLE = "polished 3D animated children's movie, bright colors, soft cinematic lighting, family-friendly"
SCRIPT = """Rayo drives along a sunny country road.
Rayo sees a small puppy waiting near a wooden bridge.
Rayo stops safely and asks the puppy what happened.
The puppy points toward a colorful kite stuck in a tree.
Rayo carefully helps the puppy get the kite back.
The puppy smiles and thanks Rayo.
Rayo drives happily toward the sunset."""
# One scene/action per line. For a 1–3 minute story, try 12–36 lines.
SEED = 42
WIDTH, HEIGHT = 512, 768
NUM_FRAMES = 121
NUM_STEPS = 8
GUIDANCE = 1.0
STORY_NAME = "meer-story"
SCRIPT_LINES = [line.strip() for line in SCRIPT.splitlines() if line.strip()]
assert SCRIPT_LINES, "Add at least one non-empty script line."
QUEUE = []
for clip_no, line in enumerate(SCRIPT_LINES, 1):
    motion_prompt = (
        f"{HERO}, {CHARACTER_DESCRIPTION}. Visual action for this scene: {line}. "
        f"{STYLE}. Keep the same proportions, colors, eyes, mouth and identifying marks. "
        "Smooth gentle motion, stable side or three-quarter view, character stays the same size, "
        "no sudden zoom, no redesign, no extra wheels or faces, clean composition, vertical 9:16."
    )
    QUEUE.append((clip_no, line, motion_prompt))
MAX_CLIPS = len(QUEUE)
STILL_FILE = "character-reference.png"
print(f"Ready: {len(QUEUE)} scenes for {HERO}; about {len(QUEUE)*5} seconds of raw visuals.")
print("First prompt preview:\n", QUEUE[0][2])

In [ ]:
# ---- GPU CHECK: what did we actually get? (T4 is a lottery on Colab free) ----
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "| total GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
# ---- INSTALL: diffusers (LTX pipeline) + helpers ----
# ---- INSTALL: PINNED versions — unpinned --upgrade drifted the env between runs ----
# Quality-pass run 1 (2026-10-25 05:23) pulled transformers 5.20.2 /
# accelerate 1.15.0 / diffusers 0.42.0 and device_map="disk" died there.
# These are the versions that run OBSERVED working — pin them so every
# run sees the same env. Do NOT re-add --upgrade. Do NOT touch torch
# (Colab ships torch 2.11.0+cu130; upgrading it breaks the runtime).
!pip install -q diffusers==0.42.0 transformers==5.20.2 accelerate==1.15.0 imageio-ffmpeg requests
print("install done")


## 1. Load LTX-Video 2B distilled (image-to-video)

**Load order (FIXED 2026-10-25 after proof run 1 — the as-written cell failed twice):**
1. **Weights first, local file:** download the OFFICIAL 0.9.8 2B distilled weights to disk (`ltxv-2b-0.9.8-distilled.safetensors`, 6,340,744,492 bytes — size-checked). NEVER pass the full `resolve/...` URL to `from_single_file` — diffusers prepends its own `resolve/main/` and 404s.
2. **No text encoder in the checkpoint:** the single file holds transformer + VAE only (944 keys, no T5). Load with `text_encoder=None, tokenizer=None` and pre-encode the prompts ONCE on CPU (T5-XXL from `PixArt-alpha/PixArt-XL-2-1024-MS`, `text_encoder/` + `tokenizer/` subfolders, bf16, `device_map="auto"` with GPU 1 GB / CPU 11 GB caps + disk offload so Colab's 12 GB RAM survives) into `prompt_embeds.pt`; the worker passes `prompt_embeds` + `prompt_attention_mask` instead of `prompt=`. (2026-10-25 05:23: `device_map="disk"` died under transformers 5.20.2 — do NOT put it back.)
3. **If the load still fails:** the cell prints the exact OFFICIAL-CLI fallback commands (clone `Lightricks/LTX-Video`, `pip install -e .[inference]`, `inference.py ...`) — do not guess, run those. (The old `multimodalart` fallback 404'd on `model_index.json` — removed.)

**VRAM survival on a 16 GB T4:** `enable_sequential_cpu_offload()` + `vae.enable_tiling()` + `vae.use_framewise_decoding = True` are MANDATORY. (Proof 1: clip 1 died in VAE decode with a 12.76 GiB OOM until framewise decoding was set — it routes decode through `_temporal_tiled_decode`.) Measured datapoint (RTX 3080 Laptop 16 GB, LTX 0.9.7 distilled, diffusers): ~257 s/clip with sequential offload, 3.8 GB VRAM — vs 2485 s / 25.6 GB without it (OOM on T4). Speed-up to try later if the proof is slow: encode the T5 text prompt on CPU once and pass `prompt_embeds` so the transformer stays on GPU without sequential offload.


In [ ]:
import gc, os, time, torch, urllib.request
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

# ---- FIX 1: download the weights to a LOCAL file first (size-checked) ----
# Passing the full https://huggingface.co/.../resolve/main/... URL to
# from_single_file FAILS: diffusers prepends its own resolve/main/ and 404s.
LTX_LOCAL = "ltxv-2b-0.9.8-distilled.safetensors"
LTX_SIZE = 6340744492  # bytes, verified on Hugging Face
t0 = time.perf_counter()
if not (os.path.exists(LTX_LOCAL) and os.path.getsize(LTX_LOCAL) == LTX_SIZE):
    print(f"Downloading LTX 2B distilled weights ({LTX_SIZE/1e9:.2f} GB)...")
    urllib.request.urlretrieve(LTX_2B_DISTILLED_URL, LTX_LOCAL)
assert os.path.getsize(LTX_LOCAL) == LTX_SIZE, "weights size mismatch — re-download"
print(f"Weights on disk: {LTX_LOCAL} ({os.path.getsize(LTX_LOCAL)} bytes, MATCH)")

# ---- FIX 2: the single-file checkpoint has NO text encoder (transformer + VAE only) ----
# Load the pipeline WITHOUT it, and pre-encode prompts separately (below).
pipe = None
load_path = None
try:
    pipe = LTXImageToVideoPipeline.from_single_file(
        LTX_LOCAL, torch_dtype=torch.bfloat16,
        text_encoder=None, tokenizer=None)
    load_path = "official 0.9.8 2B distilled (local file, text_encoder=None)"
except Exception as e:
    print("PRIMARY load failed:", type(e).__name__, str(e)[:300])
    print("OFFICIAL-CLI FALLBACK — run these cells instead:")
    print("  !git clone https://github.com/Lightricks/LTX-Video.git")
    print("  !pip install -q -e 'LTX-Video[inference]'")
    print("  !python LTX-Video/inference.py --prompt '<MOTION PROMPT>' --conditioning_media_paths bella-still.png --conditioning_start_frames 0 --height 768 --width 512 --num_frames 121 --seed 42 --pipeline_config LTX-Video/configs/ltxv-2b-0.9.8-distilled.yaml")
    raise RuntimeError("from_single_file failed — use the official CLI fallback above")

# MANDATORY on 16 GB T4: sequential CPU offload + VAE tiling + framewise decode.
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.use_framewise_decoding = True  # FIX 3: VAE decode OOMs (12.76 GiB) without this
t_load = time.perf_counter() - t0
print(f"Pipeline loaded OK via {load_path} in {t_load:.1f}s ({t_load/60:.1f} min)")

# ---- FIX 2 (continued): pre-encode ALL queue prompts ONCE on CPU ----
# Mirrors diffusers LTX _get_t5_prompt_embeds exactly (max_length 128, padding
# max_length, truncation, bf16). T5-XXL is ~19-22 GB — Colab's ~12 GB RAM
# cannot hold it, so it runs device_map="auto" (GPU 1 GB cap, CPU 11 GB cap,
# rest disk-offloaded). One-time cost; the encoder is freed before generation.
from transformers import T5EncoderModel, T5TokenizerFast
t_enc0 = time.perf_counter()
_tok = T5TokenizerFast.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-1024-MS", subfolder="tokenizer")
# FIX 2b (2026-10-25 05:23): device_map="disk" DIED under transformers 5.20.2
# (ValueError — "disk" is no longer a valid device_map string). The 5.x way:
# device_map="auto" + max_memory caps + offload_folder. GPU capped at 1 GB so
# generation keeps its VRAM; CPU RAM capped at 11 GB; the rest spills to disk.
# Encoding 3 prompts pages layers from disk — slow one-time cost, then freed.
_enc = T5EncoderModel.from_pretrained(
    "PixArt-alpha/PixArt-XL-2-1024-MS", subfolder="text_encoder",
    torch_dtype=torch.bfloat16, device_map="auto",
    max_memory={0: "1GB", "cpu": "11GB"}, offload_folder="t5-offload")
PROMPT_EMBEDS = {}
for clip_no, line, motion_prompt in QUEUE[:MAX_CLIPS]:
    _inp = _tok(motion_prompt, padding="max_length", max_length=128,
                truncation=True, add_special_tokens=True, return_tensors="pt")
    with torch.no_grad():
        _emb = _enc(_inp.input_ids)[0].to(torch.bfloat16)
    PROMPT_EMBEDS[clip_no] = (_emb, _inp.attention_mask.bool())
    print(f"  encoded clip {clip_no}: embeds {_emb.shape}")
torch.save(PROMPT_EMBEDS, "prompt_embeds.pt")
del _enc, _tok
gc.collect()
print(f"Prompts pre-encoded in {time.perf_counter()-t_enc0:.1f}s -> prompt_embeds.pt "
      f"({len(PROMPT_EMBEDS)} clips). T5 encoder freed.")


## Character reference image
Upload one clean image of the exact character you want to keep consistent. Use a front/three-quarter view, full character visible, simple background, no text or watermark. The same image conditions every scene; the prompt changes per scene.

In [ ]:
import os
from PIL import Image
from google.colab import files

if not os.path.exists(STILL_FILE):
    print("Upload the character reference image now.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No reference image uploaded.")
    chosen = next(iter(uploaded))
    os.replace(chosen, STILL_FILE)
still = Image.open(STILL_FILE).convert("RGB")
print("Reference:", STILL_FILE, still.size, os.path.getsize(STILL_FILE), "bytes")
still_small = still.resize((WIDTH, HEIGHT), Image.LANCZOS)
still_small.save("still-512x768.png")
print("Reference resized for generation:", still_small.size)

## Scene generation queue
Each line is a separate short clip. Successful clips are checkpointed locally so re-running can skip them if their staged files still exist.

In [ ]:
import base64, json, os, subprocess, time, urllib.request, urllib.error

# ---- Prompt embeds: from the load cell (same session) or prompt_embeds.pt (re-run) ----
try:
    PROMPT_EMBEDS
except NameError:
    PROMPT_EMBEDS = torch.load("prompt_embeds.pt", map_location="cpu", weights_only=True)
    print(f"Loaded prompt_embeds.pt ({len(PROMPT_EMBEDS)} clips)")

# This prototype keeps outputs in the Colab session; no token or upload required.
TOKEN = None
UPLOAD_ENABLED = False
def gh_exists(repo_path):
    return None

# ---- Checkpoint: survives a re-run in the same session; GitHub is the real checkpoint. ----
STATE_FILE = "worker-state.json"
state = {}
if os.path.exists(STATE_FILE):
    state = json.load(open(STATE_FILE))
    print("Resumed checkpoint:", {k: v.get("status") for k, v in state.items()})

def save_state():
    json.dump(state, open(STATE_FILE, "w"), indent=2)

def stage_clip(raw_path, staged_path):
    t0 = time.perf_counter()
    subprocess.run(["ffmpeg", "-y", "-i", raw_path,
                    "-vf", "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,fps=24",
                    "-an", "-c:v", "libx264", "-bf", "0", "-g", "48", "-crf", "22",
                    staged_path],
                   check=True, capture_output=True, text=True)
    return time.perf_counter() - t0

def ffprobe_check(path):
    out = subprocess.run(["ffprobe", "-v", "quiet", "-print_format", "json",
                          "-show_streams", "-show_format", path],
                         check=True, capture_output=True, text=True).stdout
    d = json.loads(out)
    s = [x for x in d["streams"] if x["codec_type"] == "video"][0]
    ok = (int(s["width"]) == 1080 and int(s["height"]) == 1920
          and s["r_frame_rate"].startswith("24") and int(s.get("has_b_frames", 1)) == 0
          and int(d["format"]["size"]) > 100000)
    print(f"  ffprobe: {s['width']}x{s['height']} fps {s['r_frame_rate']} "
          f"has_b_frames={s.get('has_b_frames')} duration={d['format']['duration']}s "
          f"size={d['format']['size']} -> {'PASS' if ok else 'FAIL'}")
    return ok

# ---- THE WORKER LOOP ----
results = []
for clip_no, line, motion_prompt in QUEUE[:MAX_CLIPS]:
    key = str(clip_no)
    repo_path = clip_repo_path(clip_no)
    # Resume: skip only if the checkpoint says done AND GitHub actually has it.
    if state.get(key, {}).get("status") == "done" and os.path.exists(f"clip-{clip_no}-staged.mp4"):
        print(f"[clip {clip_no}] SKIP — staged clip already exists locally")
        results.append(state[key])
        continue
    print(f"[clip {clip_no}] LINE: {line}")
    rec = {"clip_no": clip_no, "line": line, "status": "failed"}
    try:
        t_gen0 = time.perf_counter()
        _embeds, _mask = PROMPT_EMBEDS[clip_no]
        frames = pipe(image=still_small,
                      prompt_embeds=_embeds.to("cuda"),
                      prompt_attention_mask=_mask.to("cuda"),
                      width=WIDTH, height=HEIGHT,
                      num_frames=NUM_FRAMES,
                      num_inference_steps=NUM_STEPS,
                      guidance_scale=GUIDANCE,
                      generator=torch.Generator(device="cuda").manual_seed(SEED + clip_no),
                      ).frames[0]
        raw_path = f"clip-{clip_no}-raw.mp4"
        export_to_video(frames, raw_path, fps=24)
        rec["gen_s"] = round(time.perf_counter() - t_gen0, 1)
        print(f"  Generated {raw_path} in {rec['gen_s']:.1f}s ({rec['gen_s']/60:.1f} min), "
              f"{len(frames)} frames @24fps")

        staged_path = f"clip-{clip_no}-staged.mp4"
        rec["stage_s"] = round(stage_clip(raw_path, staged_path), 1)
        print(f"  Staged {staged_path} in {rec['stage_s']:.1f}s")
        rec["ffprobe_ok"] = ffprobe_check(staged_path)

        rec["upload_s"] = 0
        print(f"  Saved locally: {staged_path}")
        rec["total_s"] = round(rec["gen_s"] + rec["stage_s"] + rec["upload_s"], 1)
        rec["status"] = "done" if rec["ffprobe_ok"] else "done-unverified"
        print(f"  [clip {clip_no}] DONE in {rec['total_s']:.1f}s "
              f"({rec['total_s']/60:.1f} min) — status {rec['status']}")
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {str(e)[:300]}"
        print(f"  [clip {clip_no}] FAILED: {rec['error']} — moving to next line")
    state[key] = rec
    results.append(rec)
    save_state()  # checkpoint after EVERY clip — a dead session loses at most one clip

print("QUEUE DONE —", sum(1 for r in results if r.get("status", "").startswith("done")),
      "/", len(results), "clips done")


In [ ]:
# ---- Build final MP4 + free narration + subtitle file ----
import os, subprocess
from gtts import gTTS

done_clips = [f"clip-{n}-staged.mp4" for n, _, _ in QUEUE if os.path.exists(f"clip-{n}-staged.mp4")]
if not done_clips:
    raise RuntimeError("No staged scenes found. Review generation errors above.")
with open("meer-concat.txt", "w", encoding="utf-8") as f:
    for p in done_clips:
        f.write("file '" + os.path.abspath(p) + "'\n")
subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", "meer-concat.txt",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-an", "meer-visuals.mp4"],
               check=True, capture_output=True, text=True)

# gTTS uses an external free web service, not a paid API; availability can change.
TTS_LANG = "en"  # change to "hi" for Hindi
gTTS(text="\n".join(SCRIPT_LINES), lang=TTS_LANG, slow=False).save("meer-narration.mp3")
audio_probe = subprocess.run(["ffprobe", "-v", "quiet", "-show_entries", "format=duration",
                              "-of", "default=noprint_wrappers=1:nokey=1", "meer-narration.mp3"],
                             check=True, capture_output=True, text=True).stdout.strip()
audio_seconds = float(audio_probe)
subprocess.run(["ffmpeg", "-y", "-stream_loop", "-1", "-i", "meer-visuals.mp4",
                "-i", "meer-narration.mp3", "-t", str(audio_seconds),
                "-map", "0:v:0", "-map", "1:a:0", "-c:v", "libx264", "-crf", "22",
                "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "128k", "-shortest",
                "meer-final.mp4"], check=True, capture_output=True, text=True)

def srt_time(seconds):
    ms = int(round(seconds * 1000))
    h, rem = divmod(ms, 3600000); m, rem = divmod(rem, 60000); s, ms = divmod(rem, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"
with open("meer-subtitles.srt", "w", encoding="utf-8") as f:
    for i, line in enumerate(SCRIPT_LINES, 1):
        f.write(f"{i}\n{srt_time((i-1)*5)} --> {srt_time(i*5)}\n{line}\n\n")
print("FINAL OUTPUTS:")
for p in ["meer-final.mp4", "meer-narration.mp3", "meer-subtitles.srt"]:
    print(p, os.path.getsize(p), "bytes")
print("SRT timing is approximate; review before publishing.")
from google.colab import files
files.download("meer-final.mp4")
files.download("meer-narration.mp3")
files.download("meer-subtitles.srt")

## Limits and troubleshooting
- Start with 2–3 scenes. Longer stories can exhaust free GPU time or memory.
- Each line becomes a separate ~5-second generation request. The model may not depict every detail literally; revise prompts or retry scenes.
- A reference image helps but cannot guarantee perfect identity in every frame.
- gTTS is a free external web service, not a guaranteed permanent API. Subtitles are separate SRT and their timings are approximate.
- This notebook does not modify the Android APK. Integrating it requires a secure queue/worker and app-side status. Never embed tokens in the APK.
- Check current model licence and Colab free-tier rules before commercial distribution.